In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import geopandas as gpd
import os
import app
import string

# Import

In [2]:
data_folder = r"C:\Users\jsommer1\[Code]\HydrofractureShackleton_2023\OLD\data"
data_dmg_folder = r"C:\Users\jsommer1\[Code]\HydrofractureShackleton_2023\OLD\data-dmg"

In [3]:
years = ["2018", "2019", "2020"]
xmin = 2.50e6
xmax = 2.76e6
ymax = -0.215e6
ymin = -0.591e6

In [5]:
iceshelves_folder = r"greene2022_iceshelves"
all_iceshelves_files = os.listdir(data_folder)
iceshelves = [
    gpd.read_file(os.path.join(data_folder, f)).cx[xmin:xmax, ymin:ymax]
    for f in all_iceshelves_files
    if f.endswith(".shp") and any([s in f for s in years])
]
ice_buffer = 150
for ice in iceshelves:
    ice.geometry = ice.buffer(ice_buffer)
    ice = ice.dissolve()
    ice.geometry = ice.buffer(-ice_buffer)

buffer = 6e3
iceshelves_buffer = [i.buffer(buffer) for i in iceshelves]

KeyboardInterrupt: 

In [5]:
dmg_path = r"data-dmg"
target_res = 3e3

In [ ]:
dmg_files = [
    "2018_S1_30m_dmg.tif",
    "2019_S1_30m_dmg.tif",
    "2020_S1_30m_dmg.tif",
]
for i, f in enumerate(dmg_files):
    raster, transform, meta = app.geotiffs.open_resampled(
        os.path.join(dmg_path, f), target_res, mode = "average"
    )
    masked, masked_transform = app.geotiffs.mask_dataset(
        raster, transform, iceshelves_buffer[i], mode="shape", filled=False
    )
    # print(masked.shape, meta)
    # print(masked_transform)
    masked = (masked / np.nanmax(masked)).copy()
    meta["nodata"] = np.nan
    dmg_gtiff = app.geotiffs.GeoTiff(
        data = masked, metadata = meta, transform = masked_transform)
    app.geotiffs.save_geotiff(
        dmg_gtiff, os.path.join(dmg_path, f.replace("30m", f"{int(target_res):d}m").replace("_dmg.tif", "_masked_dmg.tif"))
    )

{'driver': 'GTiff', 'dtype': 'float64', 'nodata': None, 'width': 59, 'height': 176, 'count': 1, 'crs': None, 'transform': Affine(3000.0, 0.0, 2540895.0,
       0.0, -3000.0, -111075.0)}
{'driver': 'GTiff', 'dtype': 'float64', 'nodata': None, 'width': 59, 'height': 176, 'count': 1, 'crs': None, 'transform': Affine(3000.0, 0.0, 2540895.0,
       0.0, -3000.0, -111075.0)}
{'driver': 'GTiff', 'dtype': 'float64', 'nodata': None, 'width': 59, 'height': 176, 'count': 1, 'crs': None, 'transform': Affine(3000.0, 0.0, 2540895.0,
       0.0, -3000.0, -111075.0)}


In [11]:
delta_files = [
    "2018_S1_30m_delta-alpha.tif",
    "2019_S1_30m_delta-alpha.tif",
    "2020_S1_30m_delta-alpha.tif",
]

for i, f in enumerate(delta_files):
    raster, transform, meta = app.geotiffs.open(
        os.path.join(dmg_path, f)
    )

    activeness = app.geotiffs.create_active_crevasses_mask(raster, 45, 15)
    activeness_mask = np.zeros(activeness.shape) * 0
    activeness_mask[np.isfinite(activeness)] = 1
    resampled, resampled_transform, resampled_meta = app.geotiffs.resample_dataset(
        activeness_mask, transform, target_res, mode="average"
    )
    resampled[resampled == 0] = np.nan
    masked, masked_transform = app.geotiffs.mask_dataset(
        resampled, resampled_transform, iceshelves_buffer[i], mode="shape", filled=False
    )
    masked[~np.isfinite(masked)] = np.nan
    masked = masked / np.nanmax(masked)
    # print(masked.shape)
    # print(masked_transform)
    resampled_meta["nodata"] = np.nan
    print(resampled_meta)
    active_gtiff = app.geotiffs.GeoTiff(
        data = masked, metadata = resampled_meta, transform = masked_transform, crs = resampled_meta["crs"])
    app.geotiffs.save_geotiff(
        active_gtiff, os.path.join(dmg_path, f.replace("30m", f"{int(target_res):d}m").replace("_delta-alpha.tif", "_masked_activeness.tif"))
    )

{'driver': 'GTiff', 'dtype': 'float64', 'nodata': nan, 'width': 59, 'height': 176, 'count': 1, 'crs': None, 'transform': Affine(3000.0, 0.0, 2540895.0,
       0.0, -3000.0, -111075.0)}
{'driver': 'GTiff', 'dtype': 'float64', 'nodata': nan, 'width': 59, 'height': 176, 'count': 1, 'crs': None, 'transform': Affine(3000.0, 0.0, 2540895.0,
       0.0, -3000.0, -111075.0)}
{'driver': 'GTiff', 'dtype': 'float64', 'nodata': nan, 'width': 59, 'height': 176, 'count': 1, 'crs': None, 'transform': Affine(3000.0, 0.0, 2540895.0,
       0.0, -3000.0, -111075.0)}
